# 03 Inference and Final Submission
This notebook executes the final ranking step within the 5-minute compute budget using cached artifacts. It runs completely offline.

In [1]:
import subprocess, sys
pkgs = ['pandas', 'numpy', 'sentence-transformers', 'pyarrow', 'fastparquet', 'python-docx', 'scikit-learn', 'pyyaml']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-', '_').replace('python_docx','docx').replace('scikit_learn','sklearn'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All packages ready.')

/opt/homebrew/Caskroom/miniforge/base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All packages ready.


In [2]:
import os
import sys
import time
import json
import re
import subprocess
import numpy as np
import pandas as pd
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

base_dir = "/content/ai-candidate-ranking"
if not os.path.exists(base_dir):
    base_dir = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."

raw_dir = os.path.join(base_dir, "data", "raw")
processed_dir = os.path.join(base_dir, "data", "processed")
outputs_dir = os.path.join(base_dir, "outputs", "submissions")
os.makedirs(outputs_dir, exist_ok=True)

print(f"base_dir = {os.path.abspath(base_dir)}")
print(f"raw_dir  = {os.path.abspath(raw_dir)}")
assert os.path.isdir(raw_dir), f"raw_dir does not exist: {raw_dir}"
assert os.path.isdir(processed_dir), f"processed_dir does not exist: {processed_dir}"


base_dir = /Users/koustav/.gemini/antigravity-ide/scratch/ai-candidate-ranking
raw_dir  = /Users/koustav/.gemini/antigravity-ide/scratch/ai-candidate-ranking/data/raw


### 1. Load Processed Artifacts

In [3]:
print("Loading precomputed artifacts...")
start_time = time.time()

ids_path = os.path.join(processed_dir, "candidate_ids.npy")
candidate_ids = np.load(ids_path, allow_pickle=True)

emb_path = os.path.join(processed_dir, "embeddings.npy")
cand_embeddings = np.load(emb_path)

feat_path = os.path.join(processed_dir, "candidates_feather.parquet")
features_df = pd.read_parquet(feat_path)

print(f"Loaded {len(candidate_ids)} candidates.")
print(f"Loading time: {time.time() - start_time:.2f} seconds")

Loading precomputed artifacts...
Loaded 100000 candidates.
Loading time: 0.17 seconds


### 2. Build and Encode JD Text

In [4]:
def build_jd_text(docx_path):
    try:
        doc = Document(docx_path)
        return "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        print(f"Error reading JD: {e}")
        return ""

jd_path = os.path.join(raw_dir, "job_description.docx")
jd_text = build_jd_text(jd_path)

# Load model avoiding network downloads by relying on local cache
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading {model_name}...")
model = SentenceTransformer(model_name)

print("Encoding JD text...")
jd_embedding = model.encode([jd_text]) if jd_text else np.array([])

Loading sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9407.14it/s]

Encoding JD text...


### 3. Compute Semantic Similarity

In [5]:
print("Computing cosine similarities...")
sim_start = time.time()

if cand_embeddings.shape[0] > 0 and jd_embedding.shape[0] > 0:
    similarities = cosine_similarity(jd_embedding, cand_embeddings)[0]
else:
    similarities = np.zeros(len(candidate_ids))

print(f"Similarity computation took: {time.time() - sim_start:.3f} seconds")

sim_df = pd.DataFrame({"candidate_id": candidate_ids, "semantic_similarity": similarities})
full_df = pd.merge(features_df, sim_df, on="candidate_id", how="inner")

# --- Normalize features that may be out of [0,1] range ---
# github_fit_score: data is 0-100, normalize to 0-1
if full_df["github_fit_score"].max() > 1.0:
    full_df["github_fit_score"] = (full_df["github_fit_score"] / 100.0).clip(0, 1)
    print("  Normalized github_fit_score to [0,1]")

# availability_score: can exceed 1.0 due to sub-component accumulation
if full_df["availability_score"].max() > 1.0:
    max_avail = full_df["availability_score"].max()
    full_df["availability_score"] = (full_df["availability_score"] / max_avail).clip(0, 1)
    print(f"  Normalized availability_score to [0,1] (was up to {max_avail:.2f})")

# ai_skill_depth_score: unbounded accumulation, cap at max and normalize
if full_df["ai_skill_depth_score"].max() > 1.0:
    max_depth = full_df["ai_skill_depth_score"].max()
    full_df["ai_skill_depth_score"] = (full_df["ai_skill_depth_score"] / max_depth).clip(0, 1)
    print(f"  Normalized ai_skill_depth_score to [0,1] (was up to {max_depth:.2f})")

print(f"\nMerged DataFrame: {full_df.shape[0]} rows, {full_df.shape[1]} cols")


Computing cosine similarities...
Similarity computation took: 0.101 seconds
  Normalized github_fit_score to [0,1]
  Normalized availability_score to [0,1] (was up to 11.10)
  Normalized ai_skill_depth_score to [0,1] (was up to 9.94)

Merged DataFrame: 100000 rows, 16 cols


### 4. Final Scoring & Reasoning Generation

In [6]:
weights = {
    "semantic": 0.35, "role": 0.10, "product": 0.10, "exp_band": 0.10,
    "ml_years": 0.10, "ai_depth": 0.05, "avail": 0.05, "rel": 0.05,
    "github": 0.05, "geo": 0.03, "work_mode": 0.02
}
alpha = 0.5
beta = 1.0

max_ml = full_df["ml_years_estimate"].max() if "ml_years_estimate" in full_df.columns else 1.0
if max_ml == 0: max_ml = 1.0

def compute_final_score(row):
    score = (
        row.get("semantic_similarity", 0) * weights["semantic"] +
        row.get("role_title_score", 0) * weights["role"] +
        row.get("product_company_score", 0) * weights["product"] +
        row.get("experience_band_match", 0) * weights["exp_band"] +
        (row.get("ml_years_estimate", 0) / max_ml) * weights["ml_years"] +
        min(row.get("ai_skill_depth_score", 0), 1.0) * weights["ai_depth"] +
        row.get("availability_score", 0) * weights["avail"] +
        row.get("reliability_score", 0) * weights["rel"] +
        row.get("github_fit_score", 0) * weights["github"] +
        row.get("geo_fit_score", 0) * weights["geo"] +
        row.get("work_mode_fit", 0) * weights["work_mode"]
    )
    score -= (row.get("notice_period_penalty", 0) * alpha)
    score -= (row.get("honeypot_risk_score", 0) * beta)
    return max(0.0, min(1.0, score))

full_df["final_score"] = full_df.apply(compute_final_score, axis=1)

# Filter honeypots and sort
filtered_df = full_df[full_df["honeypot_risk_score"] < 0.6].copy()
ranked_df = filtered_df.sort_values(by="final_score", ascending=False).head(100).copy()

print("Extracting raw details for top candidates to generate reasoning...")
top_100_ids = set(ranked_df["candidate_id"].values)
cand_details = {}

cands_path = os.path.join(raw_dir, "candidates.jsonl")
if not os.path.exists(cands_path):
    cands_path = os.path.join(raw_dir, "sample_candidates.json")

try:
    with open(cands_path, 'r') as f:
        first_char = f.read(1)
    if first_char == '[':
        with open(cands_path, "r") as f:
            data = json.load(f)
            for cand in data:
                cid = cand.get("candidate_id")
                if cid in top_100_ids: cand_details[cid] = cand
    else:
        with open(cands_path, "r") as f:
            for line in f:
                if not line.strip(): continue
                cand = json.loads(line)
                cid = cand.get("candidate_id")
                if cid in top_100_ids: cand_details[cid] = cand
except Exception as e:
    print(f"Could not load raw data: {e}")

ai_titles = ["machine learning", "deep learning", "nlp", "search", "retrieval", 
             "embedding", "vector", "pytorch", "tensorflow", "mlflow", "llm", "ai"]
ai_pattern = re.compile("|".join(ai_titles), re.IGNORECASE)

def generate_reasoning(row):
    cid = row["candidate_id"]
    # Get title from raw candidate details (not in features DataFrame)
    cand = cand_details.get(cid, {})
    title = cand.get("profile", {}).get("current_title", "Professional") if cand else "Professional"
    yoe = row.get("total_years_experience", 0)
    ai_count = row.get("ai_core_skills_count", 0)
    sim = row.get("semantic_similarity", 0)
    
    named_skills = []
    signals = {}
    if cid in cand_details:
        cand = cand_details[cid]
        skills = cand.get("skills", [])
        signals = cand.get("redrob_signals", {})
        skill_names = [s.get("name", str(s)) if isinstance(s, dict) else str(s) for s in skills]
        named_skills = [s for s in skill_names if ai_pattern.search(s)][:3]
        
    if sim > 0.6: alignment = "exceptional alignment with JD requirements for semantic ranking"
    elif sim > 0.4: alignment = "strong background in machine learning and embeddings"
    else: alignment = "moderate exposure to AI tooling"
        
    try: resp_rate = float(signals.get("recruiter_response_rate", 0))
    except: resp_rate = 0.0
    last_act = signals.get("last_active_date", "")
    
    if resp_rate > 0.8: behavior = f"Highly responsive to recruiters ({int(resp_rate*100)}% rate)"
    elif last_act: behavior = f"Recently active on the platform ({last_act[:10]})"
    else: behavior = "Moderate platform engagement"
        
    sentence1 = f"An experienced {title} with {yoe} years of experience, demonstrating {alignment}."
    skills_str = f" Includes core AI competencies like {', '.join(named_skills)} among {ai_count} ML skills." if named_skills else f" Displays {ai_count} relevant AI skills."
    sentence2 = f" {behavior}."
    
    concern = ""
    if row.get("product_company_score", 0) < 0.3: concern = " However, background leans heavily towards IT services."
    elif row.get("notice_period_penalty", 0) > 0: concern = " Note: Long notice period."
    elif row.get("ml_years_estimate", 0) < 1.0 and ai_count > 5: concern = " Note: Many AI skills but limited explicit ML tenure."
        
    return sentence1 + skills_str + sentence2 + concern

ranked_df["reasoning"] = ranked_df.apply(generate_reasoning, axis=1)

Extracting raw details for top candidates to generate reasoning...


### 5. Format Submission

In [7]:
ranked_df["score"] = ranked_df["final_score"].round(3)
ranked_df["rank"] = range(1, len(ranked_df) + 1)
ranked_df["job_id"] = "JOB_SENIOR_AI_ENGINEER"  # Single JD for this challenge

# Enforce monotonically non-increasing scores
prev_score = ranked_df.iloc[0]["score"]
adjusted_scores = []
for s in ranked_df["score"]:
    if s > prev_score:
        adjusted_scores.append(prev_score)
    else:
        adjusted_scores.append(s)
        prev_score = s
ranked_df["score"] = adjusted_scores

submission_df = ranked_df[["candidate_id", "job_id", "rank", "score", "reasoning"]].copy()

sub_path = os.path.join(outputs_dir, "ranked_candidates.csv")
submission_df.to_csv(sub_path, index=False)
print(f"Saved final submission to {sub_path}")
print(f"Columns: {list(submission_df.columns)}")
print(f"Rows: {len(submission_df)}")


Saved final submission to ../outputs/submissions/ranked_candidates.csv
Columns: ['candidate_id', 'job_id', 'rank', 'score', 'reasoning']
Rows: 100


### 6. Validation and Inspection

In [8]:
# Locate validator script and metadata YAML
val_script = os.path.join(raw_dir, "validate_submission.py")
if not os.path.exists(val_script):
    val_script = os.path.join(base_dir, "validate_submission.py")

meta_path = os.path.join(base_dir, "submission_metadata.yaml")
if not os.path.exists(meta_path):
    meta_path = os.path.join(raw_dir, "submission_metadata_template.yaml")

print(f"Validator: {val_script} (exists={os.path.exists(val_script)})")
print(f"Metadata:  {meta_path} (exists={os.path.exists(meta_path)})")
print(f"CSV:       {sub_path} (exists={os.path.exists(sub_path)})")

print("\n=== Running Validator ===")
import subprocess
try:
    result = subprocess.run(
        [sys.executable, val_script, sub_path, meta_path],
        capture_output=True, text=True
    )
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    if result.returncode == 0:
        print("✅ Validation PASSED")
    else:
        print(f"❌ Validation FAILED (exit code {result.returncode})")
except Exception as e:
    print(f"Failed to run validator: {e}")

print("\n=== Submission Summary ===")
print(f"Columns: {list(submission_df.columns)}")
print(f"Total rows: {len(submission_df)}")
print(f"Score range: {submission_df.score.min():.4f} - {submission_df.score.max():.4f}")
print(f"Unique candidate_ids: {submission_df.candidate_id.nunique()}")

print("\n=== Top 10 rows ===")
display(submission_df.head(10))

print("\n=== 5 Random Sample rows ===")
display(submission_df.sample(min(5, len(submission_df))))


Validator: ../data/raw/validate_submission.py (exists=True)
Metadata:  ../submission_metadata.yaml (exists=True)
CSV:       ../outputs/submissions/ranked_candidates.csv (exists=True)

=== Running Validator ===


STDOUT:
Validating ../outputs/submissions/ranked_candidates.csv...
Checking metadata...
Validation successful!

✅ Validation PASSED

=== Submission Summary ===
Columns: ['candidate_id', 'job_id', 'rank', 'score', 'reasoning']
Total rows: 100
Score range: 0.6280 - 0.7360
Unique candidate_ids: 100

=== Top 10 rows ===


,candidate_id,job_id,rank,score,reasoning
81845,CAND_0081846,JOB_SENIOR_AI_ENGINEER,1,0.736,An experienced Lead AI Engineer with 6.7 years...
98845,CAND_0098846,JOB_SENIOR_AI_ENGINEER,2,0.732,An experienced AI Engineer with 7.6 years of e...
18498,CAND_0018499,JOB_SENIOR_AI_ENGINEER,3,0.718,An experienced Senior Machine Learning Enginee...
7008,CAND_0007009,JOB_SENIOR_AI_ENGINEER,4,0.713,An experienced Recommendation Systems Engineer...
80765,CAND_0080766,JOB_SENIOR_AI_ENGINEER,5,0.712,An experienced Staff Machine Learning Engineer...
83306,CAND_0083307,JOB_SENIOR_AI_ENGINEER,6,0.710,An experienced Search Engineer with 7.8 years ...
93192,CAND_0093193,JOB_SENIOR_AI_ENGINEER,7,0.698,An experienced Senior Machine Learning Enginee...
5537,CAND_0005538,JOB_SENIOR_AI_ENGINEER,8,0.688,An experienced Senior AI Engineer with 5.9 yea...
41668,CAND_0041669,JOB_SENIOR_AI_ENGINEER,9,0.684,An experienced Recommendation Systems Engineer...
87629,CAND_0087630,JOB_SENIOR_AI_ENGINEER,10,0.681,An experienced AI Engineer with 7.2 years of e...



=== 5 Random Sample rows ===


,candidate_id,job_id,rank,score,reasoning
67642,CAND_0067643,JOB_SENIOR_AI_ENGINEER,85,0.632,An experienced AI Research Engineer with 6.0 y...
50875,CAND_0050876,JOB_SENIOR_AI_ENGINEER,72,0.636,An experienced Applied ML Engineer with 6.0 ye...
86153,CAND_0086154,JOB_SENIOR_AI_ENGINEER,65,0.640,An experienced Data Scientist with 6.8 years o...
10540,CAND_0010541,JOB_SENIOR_AI_ENGINEER,80,0.634,An experienced AI Research Engineer with 6.2 y...
68350,CAND_0068351,JOB_SENIOR_AI_ENGINEER,36,0.653,An experienced Lead AI Engineer with 6.4 years...
